# Newton's Law of Cooling Simulation - Linear RNN Case

This notebook is intended to demonstrate manually setting the weights of an RNN with **linear activation** to reproduce a solution to Newton's Law of Cooling, then how to time warp to change the characteristic time scale. 

Continuous Solution:

$$
T(t) = e^{-kt}\;T_0 + (1 - e^{-kt})\;T_a
$$

Discretization Time Step with $\Delta t=1$, exact solution:

$$
T_{t+1} = e^{-k}\;T_{t} + (1 - e^{-k})\;T_a
$$

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as colors
import tensorflow as tf
import src.reproducibility as reproducibility

In [ ]:
def newton_sol(T, Ta, k, t):
    """
    Discretization of exact solution with T(0)=T0
    """
    return np.exp(-k*t)*T + (1-np.exp(-k*t))*Ta

def evolve_temp(T0, Ta, k, nsteps):
    """
    Advance the temperature forward in time using fixed time steps. 
    This function is for use with changing Ta. 
    If Ta is fixed, this will return the same as if you directly evaluated
    newton_sol for each time.
    """
    T = np.zeros(nsteps)
    T[0] = T0
    
    for t in range(1, nsteps):
        T[t] = newton_sol(T[t-1], Ta[t], k, 1)

    return T

## Simulate and Visualize Solutions

In [ ]:
nsteps=50
k1=0.2
k2=.25*2
k3=.2/2
T0 = 50
Ta = np.repeat(20, nsteps)

In [ ]:
# Confirm two different methods of calculating deterministic curves
Ts = evolve_temp(T0, Ta, k1, nsteps)
Ts2 = np.array([newton_sol(T0, Ta[0], k1, t) for t in range(0, nsteps)])
print(np.max(np.abs(Ts - Ts2))) # Should be machine-epsilon

In [ ]:
# Plot grid of constants for demonstration
kgrid = np.linspace(start=0.5, stop=0.05, num=10)
Tk = np.zeros((len(kgrid), nsteps))

fig, ax = plt.subplots(figsize=(10, 6))

# colormap and normalization
cmap = cm.viridis
norm = colors.Normalize(vmin=kgrid.min(), vmax=kgrid.max())

for i, k in enumerate(kgrid):
    Tk[i] = evolve_temp(T0, Ta, k, nsteps)
    ax.plot(Tk[i],  color=cmap(norm(k)))
del k

# colorbar
sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])  # required for older matplotlib
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Cooling Constant k")

ax.set_xlabel("Time step")
ax.set_ylabel("Temperature")
plt.show()

## Simple RNN Case

Fixed K, 1 recurrent cell

### MSE Loss can't Learn a Smooth Exponential

Networks will tend to learn to output a constant sequence corresponding to the overall signal mean. 

In [ ]:
reproducibility.set_seed(123)

inputs = tf.keras.Input(batch_shape=(None, nsteps, 1))
x = tf.keras.layers.SimpleRNN(1, return_sequences=True, activation="tanh")(inputs)
# x = tf.keras.layers.LSTM(1, return_sequences=True, activation="tanh")(inputs)
outputs = tf.keras.layers.Dense(1)(x)
rnn = tf.keras.Model(inputs, outputs)
opt = tf.keras.optimizers.Adam(learning_rate=0.1)
rnn.compile(loss = "mean_squared_error", optimizer=opt)
rnn.summary()

In [ ]:
y1 = np.array([newton_sol(T0, Ta[0], k1, t) for t in range(0, nsteps)])

N = 100  # hyperparameter: number of duplicated sequences
Xrep = np.full((N, nsteps, 1), Ta[0], dtype=np.float32)
Yrep = np.repeat(y1[None, :, None], N, axis=0)

In [ ]:
rnn.fit(Xrep, Xrep, epochs=N, verbose=False)

In [ ]:
preds = rnn.predict(Xrep[0,:,:])

In [ ]:
plt.plot(preds.flatten(), label="RNN Fit")
plt.plot(y1, label="Simulated")
plt.grid()
plt.legend()

### Manual Initialization

Setting weights of RNN without training to exactly solve the system. Just using as a method of empirical validation of scaling factors.

Discrete solution in terms of weight matrices:

\begin{aligned}
    T_{t+1} &= e^{-k}\;T_{t} + (1 - e^{-k})\;T_a\\
    &=\alpha\;T_{t} + (1 - \alpha)\;T_a, \qquad \alpha=e^{-k}\\
    &= W_h T_t + W_x T_a
\end{aligned}

Initial Hidden state: we wish to fix $T_0$, so time shift the simulation right by 1

In [ ]:
reproducibility.set_seed(123)

B = 1 # batch size
h0 = tf.fill((B, 1), T0) # Initial hidden state
inputs = tf.keras.Input(batch_shape=(None, nsteps, 1))
rnn_layer = tf.keras.layers.SimpleRNN(
    1,
    return_sequences=True,
    activation="linear"
)
x = rnn_layer(inputs, initial_state=h0)

outputs = tf.keras.layers.Dense(1)(x)
rnn = tf.keras.Model(inputs, outputs)
rnn.compile(loss = "mean_squared_error", optimizer="Adam")
rnn.summary()

In [ ]:
# Set simple RNN weights 
alpha=np.exp(-k1)

rweights = rnn.get_weights()
rweights[0] = np.array([[(1-alpha)]]) # Input
rweights[1] = np.array([[alpha]]) # Recurrent connection
rweights[2] = np.array([0])    # RNN Cell Bias
rweights[3] = np.array([[1]])  # Dense Output Activation
rweights[4] = np.array([0])  # Dense Output Bias

rnn.set_weights(rweights)

In [ ]:
X = np.full((1, nsteps, 1), Ta[0], dtype=np.float32)
pred1 = rnn.predict(X, verbose=0)
pred1 = np.concatenate(([T0], pred1.flatten()))

In [ ]:
plt.plot(pred1, '-', label="RNN Fit", linewidth=3)
plt.plot(y1, '--', label="Simulated", linewidth=3)
plt.grid()
plt.legend()

### Time Warp

Modify $W_x$ and $W_h$, leave all others unchanged

## LSTM Case

Fix k, 1 recurrent cell, 1 dense cell. 

First method: linear activation, can just manuallys set

Second method: tanh activation
- Set up weights to approximate $tanh()$ of the system
- Freeze recurrent layer, allow dense layer to learn mapping from (-1,1) to the true values.

In [ ]:
reproducibility.set_seed(123)

B = 1 # batch size
h0 = tf.zeros((B, 1))          # initial hidden state
c0 = tf.fill((B, 1), T0)       # initial cell state

inputs = tf.keras.Input(batch_shape=(None, nsteps, 1))
lstm_layer = tf.keras.layers.LSTM(
    1,
    return_sequences=True,
    activation="linear"
)
x = lstm_layer(inputs, initial_state=[h0, c0])

outputs = tf.keras.layers.Dense(1)(x)
lstm = tf.keras.Model(inputs, outputs)
lstm.compile(loss = "mean_squared_error", optimizer="Adam")
lstm.summary()

In [ ]:
alpha=np.exp(-k1)

lweights = lstm.get_weights()
# Set Input Weights, 1 feature, 4 inputs
lweights[0] = np.array([[ 
    0.0,   # input gate
    0.0,   # forget gate
    1.0,   # candidate gate
    0.0    # output gate
]])

# Set Recurrent Weights
lweights[1] = np.array([[
    0,                 # Input gate
    0,                 # Forget gate
    0,                 # Candidate
    0,                 # Output gate
]])

bf = np.log(alpha / (1 - alpha))
bi = np.log((1 - alpha) / alpha)

# Set Biases
lweights[2] = np.array([
    bi,     # input gate bias, σ(bi) = 1 - alpha
    bf,     # forget gate bias, σ(bf) = alpha
    0.0,    # candidate bias
    10.0    # output gate bias, σ approx 1
])

# Set output neuron weights(1x1)
lweights[3] = np.array([[1.0]])

# Dense bias (length-1 vector)
lweights[4] = np.array([0.0])

lstm.set_weights(lweights)

In [ ]:
X = np.full((B, nsteps, 1), Ta[0])
pred1 = lstm.predict(X, verbose=0)
pred1 = np.concatenate(([T0], pred1.flatten()))

In [ ]:
plt.plot(pred1, '-', label="LSTM Fit", linewidth=3)
plt.plot(y1, '--', label="Simulated", linewidth=3)
plt.grid()
plt.legend()

### Time warp

In [ ]:
# Warp k1 -> k2
alpha=np.exp(-k2)
bf = np.log(alpha / (1 - alpha))
bi = np.log((1 - alpha) / alpha)
# Set Biases
lweights[2] = np.array([
    bi,     # input gate bias, σ(bi) = 1 - alpha
    bf,     # forget gate bias, σ(bf) = alpha
    0.0,    # candidate bias
    10.0    # output gate bias, σ approx 1
])
lstm.set_weights(lweights)
y2 = np.array([newton_sol(T0, Ta[0], k2, t) for t in range(nsteps + 1)])
pred2 = lstm.predict(X, verbose=0)
pred2 = np.concatenate(([T0], pred2.flatten()))


# Warp k1 -> k3
alpha=np.exp(-k3)
bf = np.log(alpha / (1 - alpha))
bi = np.log((1 - alpha) / alpha)
# Set Biases
lweights[2] = np.array([
    bi,     # input gate bias, σ(bi) = 1 - alpha
    bf,     # forget gate bias, σ(bf) = alpha
    0.0,    # candidate bias
    10.0    # output gate bias, σ approx 1
])
lstm.set_weights(lweights)
y3 = np.array([newton_sol(T0, Ta[0], k3, t) for t in range(nsteps + 1)])
pred3 = lstm.predict(X, verbose=0)
pred3 = np.concatenate(([T0], pred3.flatten()))

In [ ]:
plt.plot(y1, label=f"Simulated, k={k1}", color="blue")
plt.plot(y2, label=f"Simulated, k={k2}", color="orange")
plt.plot(y3, label=f"Simulated, k={k3}", color="green")
plt.plot(pred1, label=f"LSTM, k={k1}", linestyle="dashed", color="blue")
plt.plot(pred2, label=f"LSTM, k={k2}", linestyle="dashed", color="orange")
plt.plot(pred3, label=f"LSTM, k={k3}", linestyle="dashed", color="green")
plt.legend()